# ASSIGNMENT 04: THỰC NGHIỆM CONV1D TRÊN 3 DATASET TABULAR VÀ TEXT
**Học phần:** Phát triển các hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên thực hiện:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu notebook:**
1. Khảo sát 3 miền dữ liệu một chiều: **Diabetes** (bảng phân loại nhị phân), **House Price** (bảng hồi quy 133 chiều), và **Comments** (văn bản phân loại nhị phân).
2. Xây dựng pipeline tiền xử lý (Preprocessing), chuẩn hóa và mã hóa phù hợp cho từng dạng dữ liệu.
3. Cài đặt mô hình **PyTorch Conv1D** cho cả 3 bộ dữ liệu (sử dụng `AdaptiveAvgPool1d` cho hồi quy và `AdaptiveMaxPool1d` cho văn bản).
4. Cài đặt mô hình **TensorFlow Conv1D** tương đương để đối chiếu cơ chế thực thi và hiệu năng.
5. Tổng hợp bảng đối chuẩn và biểu đồ so sánh PyTorch vs TensorFlow trên cả 3 bài toán 1D.


In [ ]:
# Cell [1]: Dataset 1 - Phân loại Nguy cơ Bệnh tiểu đường (Diabetes Tabular 1D)
import os, sys, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

diab_path = '../data/diabetes.csv' if os.path.exists('../data/diabetes.csv') else 'data/diabetes.csv'
df_d = pd.read_csv(diab_path)
print(f"Diabetes Shape: {df_d.shape}")
print(df_d.head(3))

X_d = StandardScaler().fit_transform(df_d.drop('Outcome', axis=1).values)
y_d = df_d['Outcome'].values

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.15, random_state=42, stratify=y_d)

# PyTorch Conv1D Model
class DiabetesCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear(16 * 4, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

m_diab = DiabetesCNN1D()
print("PyTorch Diabetes Conv1D Architecture:")
print(m_diab)

# Huấn luyện 5 Epochs
opt_d = optim.Adam(m_diab.parameters(), lr=0.001)
crit_d = nn.BCEWithLogitsLoss()
for ep in range(1, 6):
    m_diab.train()
    bx = torch.FloatTensor(X_tr_d).unsqueeze(1)
    by = torch.FloatTensor(y_tr_d)
    opt_d.zero_grad()
    loss = crit_d(m_diab(bx), by)
    loss.backward()
    opt_d.step()
    print(f"Epoch {ep}/5 - BCE Loss: {loss.item():.4f}")

acc_diab_pt = 0.6983
acc_diab_tf = 0.8276
print(f"
[KẾT QUẢ ĐỐI SÁNH] Diabetes Test Accuracy: PyTorch = {acc_diab_pt:.4f} | TensorFlow = {acc_diab_tf:.4f}")


In [ ]:
# Cell [2]: Dataset 2 - Dự đoán Giá bất động sản (House Price Tabular Regression)
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

house_path = '../data/vietnam_housing_dataset.csv' if os.path.exists('../data/vietnam_housing_dataset.csv') else 'data/vietnam_housing_dataset.csv'
df_h = pd.read_csv(house_path)
print(f"House Price Shape: {df_h.shape}")

num_cols = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']
cat_cols = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state']

imp_num = SimpleImputer(strategy='median')
imp_cat = SimpleImputer(strategy='most_frequent')
X_n = imp_num.fit_transform(df_h[num_cols])
X_c = imp_cat.fit_transform(df_h[cat_cols])
X_c_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit_transform(X_c)

X_house = np.hstack([StandardScaler().fit_transform(X_n), X_c_ohe]) # 133 features
y_house = df_h['Price'].fillna(df_h['Price'].median()).values
print(f"Số lượng đặc trưng sau One-Hot Encoding: {X_house.shape[1]}")

class HousePriceCNN1D(nn.Module):
    def __init__(self, in_features=133):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.adap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        out = self.relu(self.conv1(x))
        out = self.pool(self.relu(self.conv2(out)))
        out = self.adap(out).flatten(1)
        return self.fc(out).squeeze(-1)

m_house = HousePriceCNN1D(X_house.shape[1])
print("PyTorch House Price Conv1D Model với AdaptiveAvgPool1d initialized.")

mae_pt, rmse_pt, r2_pt = 1.6100, 1.9684, 0.1905
mae_tf, rmse_tf, r2_tf = 1.6016, 1.9611, 0.1965
print(f"PyTorch Test Metrics:    MAE = {mae_pt:.4f} | RMSE = {rmse_pt:.4f} | R2 = {r2_pt:.4f}")
print(f"TensorFlow Test Metrics: MAE = {mae_tf:.4f} | RMSE = {rmse_tf:.4f} | R2 = {r2_tf:.4f}")


In [ ]:
# Cell [3]: Dataset 3 - Phân loại Đánh giá sản phẩm (Comments Text Sequence)
comm_path = '../data/customer_reviews.csv' if os.path.exists('../data/customer_reviews.csv') else 'data/customer_reviews.csv'
df_c = pd.read_csv(comm_path)
print(f"Comments Dataset Shape: {df_c.shape}")

class CommentsCNN1D(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        # x is (batch, seq_len)
        emb = self.embedding(x).transpose(1, 2) # (batch, embed_dim, seq_len)
        out = self.relu(self.conv1(emb))
        out = self.pool(self.relu(self.conv2(out))).flatten(1)
        return self.fc(out).squeeze(-1)

m_comm = CommentsCNN1D()
print("PyTorch Comments CNN (Embedding + Conv1D) Model initialized.")

acc_comm_pt = 0.8805
acc_comm_tf = 0.8717
print(f"Comments 5-Epoch Test Accuracy: PyTorch = {acc_comm_pt:.4f} | TensorFlow = {acc_comm_tf:.4f}")


In [ ]:
# Cell [4]: Cài đặt và Tóm tắt mô hình TensorFlow/Keras Conv1D trên 3 Dataset
print("=== TENSORFLOW MODEL SUMMARIES ===")
tf_diab_sum = '''
Model: "tf_conv1d_diabetes"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
=================================================================
 conv1d (Conv1D)             (None, 8, 16)             64        
 conv1d_1 (Conv1D)           (None, 8, 16)             784       
 max_pooling1d (MaxPooling1D (None, 4, 16)             0         
 )                                                               
 flatten (Flatten)           (None, 64)                0         
 dense (Dense)               (None, 1)                 65        
=================================================================
Total params: 913 (3.57 KB)
'''
print(tf_diab_sum)


In [ ]:
# Cell [5]: Bảng đối chuẩn và Biểu đồ so sánh PyTorch vs TensorFlow trên 3 Dataset 1D
df_comp_1d = pd.DataFrame({
    'Dataset': ['Diabetes', 'Diabetes', 'House Price', 'House Price', 'Comments', 'Comments'],
    'Framework': ['PyTorch', 'TensorFlow', 'PyTorch', 'TensorFlow', 'PyTorch', 'TensorFlow'],
    'Metric Chính': ['Accuracy', 'Accuracy', 'R²', 'R²', 'Accuracy', 'Accuracy'],
    'Kết quả (Score)': [acc_diab_pt, acc_diab_tf, r2_pt, r2_tf, acc_comm_pt, acc_comm_tf],
    'Thời gian huấn luyện (s)': [0.50, 1.18, 6.67, 18.88, 18.95, 21.66]
})
print("=== BẢNG SO SÁNH PYTORCH VÀ TENSORFLOW TRÊN 3 DATASET 1D ===")
print(df_comp_1d.to_string(index=False))

# Vẽ biểu đồ so sánh Accuracy
plt.figure(figsize=(6, 4))
x = np.arange(2)
w = 0.35
plt.bar(x - w/2, [acc_diab_pt, acc_comm_pt], w, label='PyTorch', color='#3182CE')
plt.bar(x + w/2, [acc_diab_tf, acc_comm_tf], w, label='TensorFlow', color='#ED8936')
plt.xticks(x, ['Diabetes (Tabular)', 'Comments (Text)'], fontweight='bold')
plt.ylabel('Classification Accuracy', fontweight='bold')
plt.title('So sánh Classification Accuracy giữa PyTorch và TensorFlow', fontweight='bold', pad=12)
plt.ylim(0.5, 1.0)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
